# Phase 3: Model Training

## 🎯 Goals:
1. Load processed data from Phase 2
2. **Create scalers using DataFrames** (with column names)
3. Fit scalers on TRAINING data only
4. Transform both training and testing data
5. Train Logistic Regression models
6. Evaluate models
7. Save models and scalers
8. **Test predictions to verify everything works**

## 🔑 KEY FIXES:
- Scalers fitted on DataFrames (not arrays) → saves column names
- This prevents "X does not have valid feature names" warning
- App will use DataFrames to match training

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    recall_score, 
    precision_score, 
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 1. Load Processed Data

In [2]:
print("📂 Loading processed data...")
print("="*80)

# Load datasets
diabetes_train = pd.read_csv('../data/processed/diabetes_train.csv')
diabetes_test = pd.read_csv('../data/processed/diabetes_test.csv')
hypertension_train = pd.read_csv('../data/processed/hypertension_train.csv')
hypertension_test = pd.read_csv('../data/processed/hypertension_test.csv')

print("✓ Diabetes train:", diabetes_train.shape)
print("✓ Diabetes test:", diabetes_test.shape)
print("✓ Hypertension train:", hypertension_train.shape)
print("✓ Hypertension test:", hypertension_test.shape)

# Verify no missing values
print("\n📊 Checking for missing values...")
print(f"Diabetes train missing: {diabetes_train.isnull().sum().sum()}")
print(f"Diabetes test missing: {diabetes_test.isnull().sum().sum()}")
print(f"Hypertension train missing: {hypertension_train.isnull().sum().sum()}")
print(f"Hypertension test missing: {hypertension_test.isnull().sum().sum()}")

if (diabetes_train.isnull().sum().sum() == 0 and 
    diabetes_test.isnull().sum().sum() == 0 and
    hypertension_train.isnull().sum().sum() == 0 and
    hypertension_test.isnull().sum().sum() == 0):
    print("\n✅ All data is clean - no missing values!")
else:
    print("\n⚠️ WARNING: Found missing values! Phase 2 may not have completed properly.")

📂 Loading processed data...
✓ Diabetes train: (614, 7)
✓ Diabetes test: (154, 7)
✓ Hypertension train: (3392, 13)
✓ Hypertension test: (848, 13)

📊 Checking for missing values...
Diabetes train missing: 0
Diabetes test missing: 0
Hypertension train missing: 0
Hypertension test missing: 0

✅ All data is clean - no missing values!


## 2. Separate Features and Targets

In [3]:
print("✂️ Separating features and targets...")
print("="*80)

# Diabetes - KEEP AS DATAFRAME (this is critical!)
X_train_diabetes = diabetes_train.drop('Outcome', axis=1)
y_train_diabetes = diabetes_train['Outcome']
X_test_diabetes = diabetes_test.drop('Outcome', axis=1)
y_test_diabetes = diabetes_test['Outcome']

print("\n📊 DIABETES:")
print(f"  X_train shape: {X_train_diabetes.shape}")
print(f"  X_test shape: {X_test_diabetes.shape}")
print(f"  Features ({len(X_train_diabetes.columns)}): {X_train_diabetes.columns.tolist()}")
print(f"  Type: {type(X_train_diabetes)}")

# Hypertension - KEEP AS DATAFRAME
X_train_hypertension = hypertension_train.drop('Risk', axis=1)
y_train_hypertension = hypertension_train['Risk']
X_test_hypertension = hypertension_test.drop('Risk', axis=1)
y_test_hypertension = hypertension_test['Risk']

print("\n📊 HYPERTENSION:")
print(f"  X_train shape: {X_train_hypertension.shape}")
print(f"  X_test shape: {X_test_hypertension.shape}")
print(f"  Features ({len(X_train_hypertension.columns)}): {X_train_hypertension.columns.tolist()}")
print(f"  Type: {type(X_train_hypertension)}")

print("\n✅ SUCCESS: All features are DataFrames with column names!")

✂️ Separating features and targets...

📊 DIABETES:
  X_train shape: (614, 6)
  X_test shape: (154, 6)
  Features (6): ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']
  Type: <class 'pandas.DataFrame'>

📊 HYPERTENSION:
  X_train shape: (3392, 12)
  X_test shape: (848, 12)
  Features (12): ['male', 'age', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
  Type: <class 'pandas.DataFrame'>

✅ SUCCESS: All features are DataFrames with column names!


## 3. Create and Fit Scalers

### 🔑 CRITICAL: Fit on DataFrames (Not Arrays!)
This saves the column names in the scaler, preventing errors later.

In [4]:
print("📏 Creating scalers...")
print("="*80)

# Create Diabetes Scaler
print("\n🔧 Creating diabetes scaler...")
scaler_diabetes = StandardScaler()

# FIT on TRAINING data ONLY (as DataFrame!)
scaler_diabetes.fit(X_train_diabetes)

print(f"✓ Diabetes scaler fitted")
print(f"  Features expected: {scaler_diabetes.n_features_in_}")
print(f"  Feature names: {scaler_diabetes.feature_names_in_.tolist()}")

# Create Hypertension Scaler
print("\n🔧 Creating hypertension scaler...")
scaler_hypertension = StandardScaler()

# FIT on TRAINING data ONLY (as DataFrame!)
scaler_hypertension.fit(X_train_hypertension)

print(f"✓ Hypertension scaler fitted")
print(f"  Features expected: {scaler_hypertension.n_features_in_}")
print(f"  Feature names: {scaler_hypertension.feature_names_in_.tolist()}")

print("\n✅ Scalers created with proper feature names!")

📏 Creating scalers...

🔧 Creating diabetes scaler...
✓ Diabetes scaler fitted
  Features expected: 6
  Feature names: ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']

🔧 Creating hypertension scaler...
✓ Hypertension scaler fitted
  Features expected: 12
  Feature names: ['male', 'age', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

✅ Scalers created with proper feature names!


## 4. Scale Data

In [5]:
print("⚖️ Scaling data...")
print("="*80)

# Scale Diabetes data
print("\n🔧 Scaling diabetes data...")
X_train_diabetes_scaled = scaler_diabetes.transform(X_train_diabetes)
X_test_diabetes_scaled = scaler_diabetes.transform(X_test_diabetes)

print(f"✓ Diabetes train scaled: {X_train_diabetes_scaled.shape}")
print(f"✓ Diabetes test scaled: {X_test_diabetes_scaled.shape}")
print(f"  Sample scaled values (first row): {X_train_diabetes_scaled[0][:3]}...")

# Scale Hypertension data
print("\n🔧 Scaling hypertension data...")
X_train_hypertension_scaled = scaler_hypertension.transform(X_train_hypertension)
X_test_hypertension_scaled = scaler_hypertension.transform(X_test_hypertension)

print(f"✓ Hypertension train scaled: {X_train_hypertension_scaled.shape}")
print(f"✓ Hypertension test scaled: {X_test_hypertension_scaled.shape}")
print(f"  Sample scaled values (first row): {X_train_hypertension_scaled[0][:3]}...")

print("\n✅ All data scaled successfully!")
print("\n💡 Note: Scaled values are small numbers (around -2 to +2) - this is correct!")

⚖️ Scaling data...

🔧 Scaling diabetes data...
✓ Diabetes train scaled: (614, 6)
✓ Diabetes test scaled: (154, 6)
  Sample scaled values (first row): [-0.85135507 -1.0575301  -0.82797293]...

🔧 Scaling hypertension data...
✓ Hypertension train scaled: (3392, 12)
✓ Hypertension test scaled: (848, 12)
  Sample scaled values (first row): [1.16498605 0.98726646 1.02386971]...

✅ All data scaled successfully!

💡 Note: Scaled values are small numbers (around -2 to +2) - this is correct!


## 5. Train Models

In [6]:
print("🤖 Training models...")
print("="*80)

# Train Diabetes Model
print("\n🔧 Training diabetes model...")
diabetes_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  # Handle class imbalance
)

diabetes_model.fit(X_train_diabetes_scaled, y_train_diabetes)
print("✓ Diabetes model trained")

# Train Hypertension Model
print("\n🔧 Training hypertension model...")
hypertension_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)

hypertension_model.fit(X_train_hypertension_scaled, y_train_hypertension)
print("✓ Hypertension model trained")

print("\n✅ Both models trained successfully!")

🤖 Training models...

🔧 Training diabetes model...
✓ Diabetes model trained

🔧 Training hypertension model...
✓ Hypertension model trained

✅ Both models trained successfully!


## 6. Evaluate Models

In [7]:
print("📊 Evaluating models...")
print("="*80)

# Evaluate Diabetes Model
print("\n📈 DIABETES MODEL PERFORMANCE:")
print("-" * 50)

y_pred_diabetes = diabetes_model.predict(X_test_diabetes_scaled)
y_pred_proba_diabetes = diabetes_model.predict_proba(X_test_diabetes_scaled)[:, 1]

diabetes_accuracy = accuracy_score(y_test_diabetes, y_pred_diabetes)
diabetes_recall = recall_score(y_test_diabetes, y_pred_diabetes)
diabetes_precision = precision_score(y_test_diabetes, y_pred_diabetes)
diabetes_f1 = f1_score(y_test_diabetes, y_pred_diabetes)
diabetes_auc = roc_auc_score(y_test_diabetes, y_pred_proba_diabetes)

print(f"Accuracy:  {diabetes_accuracy*100:.1f}%")
print(f"Recall:    {diabetes_recall*100:.1f}%")
print(f"Precision: {diabetes_precision*100:.1f}%")
print(f"F1 Score:  {diabetes_f1*100:.1f}%")
print(f"AUC:       {diabetes_auc:.3f}")

# Check if metrics are good
if diabetes_accuracy >= 0.75 and diabetes_recall >= 0.70:
    print("\n✅ Diabetes model meets performance targets!")
else:
    print("\n⚠️ Diabetes model performance below targets (Accuracy 75%, Recall 70%)")

# Evaluate Hypertension Model
print("\n📈 HYPERTENSION MODEL PERFORMANCE:")
print("-" * 50)

y_pred_hypertension = hypertension_model.predict(X_test_hypertension_scaled)
y_pred_proba_hypertension = hypertension_model.predict_proba(X_test_hypertension_scaled)[:, 1]

hypertension_accuracy = accuracy_score(y_test_hypertension, y_pred_hypertension)
hypertension_recall = recall_score(y_test_hypertension, y_pred_hypertension)
hypertension_precision = precision_score(y_test_hypertension, y_pred_hypertension)
hypertension_f1 = f1_score(y_test_hypertension, y_pred_hypertension)
hypertension_auc = roc_auc_score(y_test_hypertension, y_pred_proba_hypertension)

print(f"Accuracy:  {hypertension_accuracy*100:.1f}%")
print(f"Recall:    {hypertension_recall*100:.1f}%")
print(f"Precision: {hypertension_precision*100:.1f}%")
print(f"F1 Score:  {hypertension_f1*100:.1f}%")
print(f"AUC:       {hypertension_auc:.3f}")

# Check if metrics are good
if hypertension_accuracy >= 0.78 and hypertension_recall >= 0.75:
    print("\n✅ Hypertension model meets performance targets!")
else:
    print("\n⚠️ Hypertension model performance below targets (Accuracy 78%, Recall 75%)")

📊 Evaluating models...

📈 DIABETES MODEL PERFORMANCE:
--------------------------------------------------
Accuracy:  73.4%
Recall:    70.4%
Precision: 60.3%
F1 Score:  65.0%
AUC:       0.811

⚠️ Diabetes model performance below targets (Accuracy 75%, Recall 70%)

📈 HYPERTENSION MODEL PERFORMANCE:
--------------------------------------------------
Accuracy:  87.4%
Recall:    89.0%
Precision: 75.0%
F1 Score:  81.4%
AUC:       0.951

✅ Hypertension model meets performance targets!


## 7. Save Models and Scalers

In [8]:
print("\n💾 Saving models and scalers...")
print("="*80)

# Create models directory
os.makedirs('../models', exist_ok=True)

# Save Diabetes model and scaler
joblib.dump(diabetes_model, '../models/diabetes_model.pkl')
joblib.dump(scaler_diabetes, '../models/scaler_diabetes.pkl')
print("✓ Diabetes model saved: ../models/diabetes_model.pkl")
print("✓ Diabetes scaler saved: ../models/scaler_diabetes.pkl")

# Save Hypertension model and scaler
joblib.dump(hypertension_model, '../models/hypertension_model.pkl')
joblib.dump(scaler_hypertension, '../models/scaler_hypertension.pkl')
print("✓ Hypertension model saved: ../models/hypertension_model.pkl")
print("✓ Hypertension scaler saved: ../models/scaler_hypertension.pkl")

print("\n✅ All models and scalers saved successfully!")


💾 Saving models and scalers...
✓ Diabetes model saved: ../models/diabetes_model.pkl
✓ Diabetes scaler saved: ../models/scaler_diabetes.pkl
✓ Hypertension model saved: ../models/hypertension_model.pkl
✓ Hypertension scaler saved: ../models/scaler_hypertension.pkl

✅ All models and scalers saved successfully!


## 8. TEST PREDICTIONS

### 🧪 This is CRITICAL - We test that predictions work correctly!

In [9]:
print("\n🧪 TESTING PREDICTIONS...")
print("="*80)

# Test 1: Low Risk Diabetes Profile
print("\n📊 TEST 1: Healthy Diabetes Profile (Should be LOW RISK)")
print("-" * 50)

test_diabetes_low = pd.DataFrame([[
    0,    # Pregnancies
    80,   # Glucose (healthy)
    70,   # BloodPressure (healthy)
    22,   # BMI (healthy)
    0.1,  # DiabetesPedigreeFunction (low family history)
    25    # Age (young)
]], columns=X_train_diabetes.columns)

print("Input values:")
print(test_diabetes_low)

# Scale and predict
test_scaled = scaler_diabetes.transform(test_diabetes_low)
prediction = diabetes_model.predict(test_scaled)[0]
probability = diabetes_model.predict_proba(test_scaled)[0][1]

print(f"\nPrediction: {prediction} (0=No diabetes, 1=Diabetes)")
print(f"Probability: {probability*100:.1f}%")

if probability < 0.30:
    print("✅ CORRECT: Shows LOW RISK as expected!")
else:
    print(f"⚠️ WARNING: Should be low risk but shows {probability*100:.1f}%")

# Test 2: High Risk Diabetes Profile
print("\n📊 TEST 2: High Risk Diabetes Profile (Should be HIGH RISK)")
print("-" * 50)

test_diabetes_high = pd.DataFrame([[
    6,    # Pregnancies (many)
    180,  # Glucose (very high)
    90,   # BloodPressure (high)
    35,   # BMI (obese)
    0.8,  # DiabetesPedigreeFunction (strong family history)
    55    # Age (older)
]], columns=X_train_diabetes.columns)

print("Input values:")
print(test_diabetes_high)

test_scaled = scaler_diabetes.transform(test_diabetes_high)
prediction = diabetes_model.predict(test_scaled)[0]
probability = diabetes_model.predict_proba(test_scaled)[0][1]

print(f"\nPrediction: {prediction}")
print(f"Probability: {probability*100:.1f}%")

if probability > 0.60:
    print("✅ CORRECT: Shows HIGH RISK as expected!")
else:
    print(f"⚠️ WARNING: Should be high risk but shows {probability*100:.1f}%")

# Test 3: Low Risk Hypertension Profile
print("\n📊 TEST 3: Healthy Hypertension Profile (Should be LOW RISK)")
print("-" * 50)

test_hypertension_low = pd.DataFrame([[
    0,    # male (female)
    25,   # age (young)
    0,    # currentSmoker (no)
    0,    # cigsPerDay
    0,    # BPMeds (no)
    0,    # diabetes (no)
    180,  # totChol (healthy)
    110,  # sysBP (healthy)
    70,   # diaBP (healthy)
    21,   # BMI (healthy)
    70,   # heartRate (healthy)
    85    # glucose (healthy)
]], columns=X_train_hypertension.columns)

print("Input values:")
print(test_hypertension_low)

test_scaled = scaler_hypertension.transform(test_hypertension_low)
prediction = hypertension_model.predict(test_scaled)[0]
probability = hypertension_model.predict_proba(test_scaled)[0][1]

print(f"\nPrediction: {prediction}")
print(f"Probability: {probability*100:.1f}%")

if probability < 0.30:
    print("✅ CORRECT: Shows LOW RISK as expected!")
else:
    print(f"⚠️ WARNING: Should be low risk but shows {probability*100:.1f}%")


🧪 TESTING PREDICTIONS...

📊 TEST 1: Healthy Diabetes Profile (Should be LOW RISK)
--------------------------------------------------
Input values:
   Pregnancies  Glucose  BloodPressure  BMI  DiabetesPedigreeFunction  Age
0            0       80             70   22                       0.1   25

Prediction: 0 (0=No diabetes, 1=Diabetes)
Probability: 2.1%
✅ CORRECT: Shows LOW RISK as expected!

📊 TEST 2: High Risk Diabetes Profile (Should be HIGH RISK)
--------------------------------------------------
Input values:
   Pregnancies  Glucose  BloodPressure  BMI  DiabetesPedigreeFunction  Age
0            6      180             90   35                       0.8   55

Prediction: 1
Probability: 95.8%
✅ CORRECT: Shows HIGH RISK as expected!

📊 TEST 3: Healthy Hypertension Profile (Should be LOW RISK)
--------------------------------------------------
Input values:
   male  age  currentSmoker  cigsPerDay  BPMeds  diabetes  totChol  sysBP  \
0     0   25              0           0       0   

## 9. Final Verification

In [10]:
print("\n" + "="*80)
print("✅ PHASE 3 COMPLETE - FINAL VERIFICATION")
print("="*80)

# Reload models to verify they saved correctly
diabetes_model_check = joblib.load('../models/diabetes_model.pkl')
scaler_diabetes_check = joblib.load('../models/scaler_diabetes.pkl')
hypertension_model_check = joblib.load('../models/hypertension_model.pkl')
scaler_hypertension_check = joblib.load('../models/scaler_hypertension.pkl')

print("\n📊 DIABETES MODEL:")
print(f"  Scaler features: {scaler_diabetes_check.n_features_in_}")
print(f"  Feature names: {scaler_diabetes_check.feature_names_in_.tolist()}")
print(f"  Model accuracy: {diabetes_accuracy*100:.1f}%")

print("\n📊 HYPERTENSION MODEL:")
print(f"  Scaler features: {scaler_hypertension_check.n_features_in_}")
print(f"  Feature names: {scaler_hypertension_check.feature_names_in_.tolist()}")
print(f"  Model accuracy: {hypertension_accuracy*100:.1f}%")

print("\n" + "="*80)
print("✅✅✅ SUCCESS! Models trained and saved correctly!")
print("="*80)

print("\n🚀 Ready to run the app!")
print("\n📝 Next steps:")
print("  1. Navigate to project root directory")
print("  2. Run: py -3.12 app/main.py")
print("  3. Test with the profiles above")
print("\n💡 The app will now work correctly with NO warnings!")


✅ PHASE 3 COMPLETE - FINAL VERIFICATION

📊 DIABETES MODEL:
  Scaler features: 6
  Feature names: ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']
  Model accuracy: 73.4%

📊 HYPERTENSION MODEL:
  Scaler features: 12
  Feature names: ['male', 'age', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
  Model accuracy: 87.4%

✅✅✅ SUCCESS! Models trained and saved correctly!

🚀 Ready to run the app!

📝 Next steps:
  1. Navigate to project root directory
  2. Run: py -3.12 app/main.py
  3. Test with the profiles above

💡 The app will now work correctly with NO warnings!
